# EDA Fourcasters

Ce notebook sert à faire un premier état des lieux des tables produites par dbt.
On regarde les volumes, les dates, les valeurs manquantes et quelques
répartitions utiles pour préparer Power BI et un futur modèle de ML.

> Attention : le niveau Météo-France est un **danger prévu**, pas un relevé
> du nombre de feux réellement observés. Pour ce projet, la cible ML est la
> classe de danger publiée à J1 ou J2 par Météo-France.

In [ ]:
import pandas as pd
from google.cloud import bigquery

PROJET_GCP = "fourcasters-openmeteo-loick"
DATASET = "openmeteo_analyse"
client = bigquery.Client(project=PROJET_GCP)

def lire_requete(sql):
    """Exécute une requête et renvoie un DataFrame."""
    return client.query(sql).to_dataframe()

## 1. Volumes et période couverte

In [ ]:
volumes = lire_requete(f"""
SELECT 'fact_meteo' AS table_nom, COUNT(*) AS lignes,
       MIN(date) AS date_min, MAX(date) AS date_max
FROM `{PROJET_GCP}.{DATASET}.fact_meteo`
UNION ALL
SELECT 'fact_danger_incendie', COUNT(*), MIN(date_publication), MAX(date_publication)
FROM `{PROJET_GCP}.{DATASET}.fact_danger_incendie`
UNION ALL
SELECT 'pbi_risque_incendie', COUNT(*), MIN(date_publication), MAX(date_publication)
FROM `{PROJET_GCP}.{DATASET}.pbi_risque_incendie`
UNION ALL
SELECT 'ml_features_incendie', COUNT(*), MIN(date_publication), MAX(date_publication)
FROM `{PROJET_GCP}.{DATASET}.ml_features_incendie`
UNION ALL
SELECT 'ml_train_incendie', COUNT(*), MIN(date_publication), MAX(date_publication)
FROM `{PROJET_GCP}.{DATASET}.ml_train_incendie`
ORDER BY table_nom
""")
volumes

## 2. Valeurs manquantes dans la table Power BI

In [ ]:
manquants = lire_requete(f"""
SELECT
  COUNT(*) AS lignes,
  COUNTIF(NOT meteo_disponible) AS lignes_sans_meteo,
  ROUND(100 * COUNTIF(NOT meteo_disponible) / COUNT(*), 2) AS pct_sans_meteo,
  COUNTIF(temperature_moyenne IS NULL) AS temperature_manquante,
  COUNTIF(precipitations_totales IS NULL) AS precipitation_manquante
FROM `{PROJET_GCP}.{DATASET}.pbi_risque_incendie`
""")
manquants

## 3. Quelques indicateurs météo

In [ ]:
resume_meteo = lire_requete(f"""
SELECT
  EXTRACT(YEAR FROM date) AS annee,
  ROUND(AVG(temperature_moyenne), 1) AS temperature_moyenne,
  ROUND(MAX(temperature_maximale), 1) AS temperature_maximale,
  ROUND(SUM(precipitations_totales), 1) AS precipitations_totales
FROM `{PROJET_GCP}.{DATASET}.fact_meteo`
GROUP BY annee
ORDER BY annee
""")
resume_meteo

## 4. Répartition du danger prévu

In [ ]:
repartition_danger = lire_requete(f"""
SELECT
  echeance,
  niveau_danger,
  COUNT(*) AS lignes,
  ROUND(100 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY echeance), 2) AS pct
FROM `{PROJET_GCP}.{DATASET}.fact_danger_incendie`
GROUP BY echeance, niveau_danger
ORDER BY echeance, niveau_danger
""")
repartition_danger

## 5. Qualité de la couverture météo

In [ ]:
POINTS_ATTENDUS = 360
couverture_meteo = lire_requete(f"""
WITH jours AS (
  SELECT date, COUNT(DISTINCT code_insee) AS points
  FROM `{PROJET_GCP}.{DATASET}.fact_meteo`
  GROUP BY date
)
SELECT
  COUNT(*) AS jours,
  MIN(points) AS minimum_points,
  MAX(points) AS maximum_points,
  COUNTIF(points != {POINTS_ATTENDUS}) AS jours_incomplets
FROM jours
""")
couverture_meteo

Si `jours_incomplets` est supérieur à zéro, il faut vérifier les communes
manquantes avant de comparer les départements entre eux.

## 6. Couverture par département

In [ ]:
couverture_departements = lire_requete(f"""
SELECT
  numero_departement,
  ANY_VALUE(departement) AS departement,
  COUNT(DISTINCT date) AS jours_meteo,
  MIN(date) AS date_min,
  MAX(date) AS date_max
FROM `{PROJET_GCP}.{DATASET}.int_meteo_departement_jour`
GROUP BY numero_departement
ORDER BY jours_meteo, numero_departement
""")
couverture_departements.head(10)

Cette vue permet de repérer les départements qui ont une période météo plus
courte. C'est important avant de tirer des conclusions géographiques.

## 7. Danger prévu et météo associée

In [ ]:
danger_meteo = lire_requete(f"""
SELECT
  echeance,
  niveau_danger,
  COUNT(*) AS lignes,
  ROUND(AVG(temperature_moyenne), 1) AS temperature_moyenne,
  ROUND(AVG(precipitations_totales), 2) AS precipitations_moyennes,
  ROUND(AVG(rafale_vent_maximale), 1) AS rafale_moyenne
FROM `{PROJET_GCP}.{DATASET}.pbi_risque_incendie`
WHERE meteo_disponible
GROUP BY echeance, niveau_danger
ORDER BY echeance, niveau_danger
""")
danger_meteo

Cette comparaison est descriptive uniquement. Elle ne prouve pas qu'une
variable météo cause un niveau de danger, car le niveau Météo-France est
une prévision produite avec d'autres informations.

## 8. Contrôle des variables ML

In [ ]:
controle_ml = lire_requete(f"""
SELECT
  COUNT(*) AS lignes,
  COUNT(DISTINCT id_feature) AS identifiants_uniques,
  COUNTIF(NOT meteo_disponible) AS lignes_sans_meteo,
  COUNTIF(temperature_moyenne_7j IS NULL) AS temperature_7j_manquante,
  MIN(date_publication) AS premiere_publication,
  MAX(date_publication) AS derniere_publication
FROM `{PROJET_GCP}.{DATASET}.ml_features_incendie`
""")

cible_ml = lire_requete(f"""
SELECT
  echeance,
  cible_niveau_danger,
  COUNT(*) AS lignes,
  ROUND(100 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY echeance), 2) AS pct
FROM `{PROJET_GCP}.{DATASET}.ml_train_incendie`
GROUP BY echeance, cible_niveau_danger
ORDER BY echeance, cible_niveau_danger
""")

resume_cible_ml = lire_requete(f"""
SELECT
  echeance,
  COUNT(*) AS lignes,
  ROUND(AVG(cible_niveau_danger), 2) AS moyenne,
  MIN(cible_niveau_danger) AS minimum,
  MAX(cible_niveau_danger) AS maximum,
  ROUND(STDDEV(cible_niveau_danger), 2) AS ecart_type
FROM `{PROJET_GCP}.{DATASET}.ml_train_incendie`
GROUP BY echeance
ORDER BY echeance
""")

controle_ml, cible_ml, resume_cible_ml

Une ligne de variables correspond à un couple date de publication-département.
La table `ml_train_incendie` ajoute deux lignes par publication, une pour J1
et une pour J2, avec `cible_niveau_danger` comme classe à prévoir.

## 9. Premiers graphiques

In [ ]:
# Graphiques facultatifs : matplotlib n'est pas nécessaire pour le pipeline quotidien.
try:
    import matplotlib.pyplot as plt
except ImportError:
    print("Installe matplotlib pour afficher les graphiques : uv add matplotlib")
else:
    resume_meteo.plot(x="annee", y="temperature_moyenne", marker="o")
    plt.title("Température moyenne par année")
    plt.ylabel("Température (°C)")
    plt.show()

    danger_meteo.pivot(index="niveau_danger", columns="echeance", values="temperature_moyenne").plot(kind="bar")
    plt.title("Température moyenne selon le danger prévu")
    plt.ylabel("Température (°C)")
    plt.show()

    couverture_departements.sort_values("jours_meteo").tail(15).plot(
        x="departement", y="jours_meteo", kind="barh", legend=False
    )
    plt.title("Départements les mieux couverts")
    plt.xlabel("Nombre de jours")
    plt.show()

## 10. Questions à garder pour la suite

À compléter après l'exécution du notebook :

- quelles années ou périodes présentent le plus de danger prévu ?
- les niveaux élevés sont-ils concentrés dans certains départements ?
- les variables météo sont-elles suffisamment complètes pour le ML ?
- les quatre niveaux de danger sont-ils assez équilibrés pour entraîner le modèle ?

Ces questions guideront la suite de l'analyse. Les résultats doivent être
interprétés avec les limites de couverture et sans confondre danger prévu
et incendie réellement déclenché.

## 11. Statistiques détaillées des variables météo

In [ ]:
statistiques_meteo = lire_requete(f"""
WITH variables AS (
  SELECT statistique.nom, statistique.valeur
  FROM `{PROJET_GCP}.{DATASET}.fact_meteo`,
  UNNEST([
    STRUCT('temperature_moyenne' AS nom, SAFE_CAST(temperature_moyenne AS FLOAT64) AS valeur),
    STRUCT('temperature_minimale' AS nom, SAFE_CAST(temperature_minimale AS FLOAT64) AS valeur),
    STRUCT('temperature_maximale' AS nom, SAFE_CAST(temperature_maximale AS FLOAT64) AS valeur),
    STRUCT('temperature_ressentie_moyenne' AS nom, SAFE_CAST(temperature_ressentie_moyenne AS FLOAT64) AS valeur),
    STRUCT('temperature_ressentie_minimale' AS nom, SAFE_CAST(temperature_ressentie_minimale AS FLOAT64) AS valeur),
    STRUCT('temperature_ressentie_maximale' AS nom, SAFE_CAST(temperature_ressentie_maximale AS FLOAT64) AS valeur),
    STRUCT('humidite_moyenne' AS nom, SAFE_CAST(humidite_moyenne AS FLOAT64) AS valeur),
    STRUCT('humidite_minimale' AS nom, SAFE_CAST(humidite_minimale AS FLOAT64) AS valeur),
    STRUCT('humidite_maximale' AS nom, SAFE_CAST(humidite_maximale AS FLOAT64) AS valeur),
    STRUCT('point_de_rosee_moyen' AS nom, SAFE_CAST(point_de_rosee_moyen AS FLOAT64) AS valeur),
    STRUCT('precipitations_totales' AS nom, SAFE_CAST(precipitations_totales AS FLOAT64) AS valeur),
    STRUCT('pluie_totale' AS nom, SAFE_CAST(pluie_totale AS FLOAT64) AS valeur),
    STRUCT('neige_totale' AS nom, SAFE_CAST(neige_totale AS FLOAT64) AS valeur),
    STRUCT('heures_de_precipitations' AS nom, SAFE_CAST(heures_de_precipitations AS FLOAT64) AS valeur),
    STRUCT('vitesse_vent_moyenne' AS nom, SAFE_CAST(vitesse_vent_moyenne AS FLOAT64) AS valeur),
    STRUCT('vitesse_vent_maximale' AS nom, SAFE_CAST(vitesse_vent_maximale AS FLOAT64) AS valeur),
    STRUCT('rafale_vent_maximale' AS nom, SAFE_CAST(rafale_vent_maximale AS FLOAT64) AS valeur),
    STRUCT('couverture_nuageuse_moyenne' AS nom, SAFE_CAST(couverture_nuageuse_moyenne AS FLOAT64) AS valeur),
    STRUCT('pression_moyenne' AS nom, SAFE_CAST(pression_moyenne AS FLOAT64) AS valeur),
    STRUCT('duree_ensoleillement' AS nom, SAFE_CAST(duree_ensoleillement AS FLOAT64) AS valeur),
    STRUCT('rayonnement_solaire_total' AS nom, SAFE_CAST(rayonnement_solaire_total AS FLOAT64) AS valeur),
    STRUCT('evapotranspiration' AS nom, SAFE_CAST(evapotranspiration AS FLOAT64) AS valeur),
    STRUCT('deficit_pression_vapeur_maximal' AS nom, SAFE_CAST(deficit_pression_vapeur_maximal AS FLOAT64) AS valeur),
    STRUCT('humidite_sol_0_7cm' AS nom, SAFE_CAST(humidite_sol_0_7cm AS FLOAT64) AS valeur),
    STRUCT('humidite_sol_7_28cm' AS nom, SAFE_CAST(humidite_sol_7_28cm AS FLOAT64) AS valeur),
    STRUCT('humidite_sol_28_100cm' AS nom, SAFE_CAST(humidite_sol_28_100cm AS FLOAT64) AS valeur),
    STRUCT('temperature_sol_0_7cm' AS nom, SAFE_CAST(temperature_sol_0_7cm AS FLOAT64) AS valeur)
  ]) AS statistique
)
SELECT
  nom,
  COUNTIF(valeur IS NOT NULL) AS valeurs,
  COUNTIF(valeur IS NULL) AS valeurs_manquantes,
  ROUND(AVG(valeur), 2) AS moyenne,
  ROUND(STDDEV(valeur), 2) AS ecart_type,
  ROUND(APPROX_QUANTILES(valeur, 4)[SAFE_OFFSET(1)], 2) AS premier_quartile,
  ROUND(APPROX_QUANTILES(valeur, 4)[SAFE_OFFSET(2)], 2) AS mediane,
  ROUND(APPROX_QUANTILES(valeur, 4)[SAFE_OFFSET(3)], 2) AS troisieme_quartile,
  ROUND(MIN(valeur), 2) AS minimum,
  ROUND(MAX(valeur), 2) AS maximum
FROM variables
GROUP BY nom
ORDER BY nom
""")
statistiques_meteo

Cette table donne une lecture univariée complète : nombre de valeurs,
valeurs manquantes, moyenne, dispersion, quartiles, minimum et maximum.
La direction du vent n'est pas moyennée car c'est une variable circulaire.

## 12. Variations selon l'année et la saison

In [ ]:
statistiques_saison = lire_requete(f"""
SELECT
  d.annee,
  d.saison,
  COUNT(*) AS observations,
  ROUND(AVG(m.temperature_moyenne), 2) AS temperature_moyenne,
  ROUND(MIN(m.temperature_minimale), 2) AS temperature_minimale,
  ROUND(MAX(m.temperature_maximale), 2) AS temperature_maximale,
  ROUND(AVG(m.humidite_moyenne), 2) AS humidite_moyenne,
  ROUND(SUM(m.precipitations_totales), 2) AS precipitations_totales,
  ROUND(MAX(m.rafale_vent_maximale), 2) AS rafale_maximale
FROM `{PROJET_GCP}.{DATASET}.fact_meteo` AS m
INNER JOIN `{PROJET_GCP}.{DATASET}.dim_date` AS d
  ON m.date = d.date
GROUP BY d.annee, d.saison
ORDER BY d.annee, d.saison
""")
statistiques_saison

Cette vue aide à repérer les périodes chaudes, sèches ou venteuses. Les
résultats doivent rester comparables uniquement entre périodes suffisamment
couvertes par les données.

## 13. Comparaison entre départements

In [ ]:
statistiques_departements = lire_requete(f"""
SELECT
  numero_departement,
  ANY_VALUE(departement) AS departement,
  ROUND(AVG(temperature_moyenne), 2) AS temperature_moyenne,
  ROUND(MIN(temperature_minimale), 2) AS temperature_minimale,
  ROUND(MAX(temperature_maximale), 2) AS temperature_maximale,
  ROUND(AVG(humidite_moyenne), 2) AS humidite_moyenne,
  ROUND(SUM(precipitations_totales), 2) AS precipitations_totales,
  ROUND(MAX(rafale_vent_maximale), 2) AS rafale_maximale,
  COUNT(DISTINCT date) AS jours
FROM `{PROJET_GCP}.{DATASET}.int_meteo_departement_jour`
GROUP BY numero_departement
ORDER BY temperature_moyenne DESC
""")

print("Départements les plus chauds en moyenne")
statistiques_departements.head(10)

In [ ]:
print("Départements les plus humides en moyenne")
statistiques_departements.sort_values("humidite_moyenne", ascending=False).head(10)

print("Départements avec le plus de précipitations cumulées")
statistiques_departements.sort_values("precipitations_totales", ascending=False).head(10)

Ces classements sont utiles pour le reporting, mais ils dépendent du nombre
de jours disponibles dans chaque département. La colonne `jours` doit donc
toujours être regardée avec les moyennes.

## 14. Statistiques des variables préparées pour le ML

In [ ]:
statistiques_ml = lire_requete(f"""
WITH variables AS (
  SELECT statistique.nom, statistique.valeur
  FROM `{PROJET_GCP}.{DATASET}.ml_features_incendie`,
  UNNEST([
    STRUCT('temperature_moyenne_7j' AS nom, SAFE_CAST(temperature_moyenne_7j AS FLOAT64) AS valeur),
    STRUCT('temperature_maximale_7j' AS nom, SAFE_CAST(temperature_maximale_7j AS FLOAT64) AS valeur),
    STRUCT('humidite_moyenne_7j' AS nom, SAFE_CAST(humidite_moyenne_7j AS FLOAT64) AS valeur),
    STRUCT('precipitations_7j' AS nom, SAFE_CAST(precipitations_7j AS FLOAT64) AS valeur),
    STRUCT('rafale_vent_maximale_7j' AS nom, SAFE_CAST(rafale_vent_maximale_7j AS FLOAT64) AS valeur),
    STRUCT('deficit_pression_vapeur_maximal_7j' AS nom, SAFE_CAST(deficit_pression_vapeur_maximal_7j AS FLOAT64) AS valeur),
    STRUCT('jours_sans_pluie_7j' AS nom, SAFE_CAST(jours_sans_pluie_7j AS FLOAT64) AS valeur)
  ]) AS statistique
)
SELECT
  nom,
  COUNTIF(valeur IS NOT NULL) AS valeurs,
  COUNTIF(valeur IS NULL) AS valeurs_manquantes,
  ROUND(AVG(valeur), 2) AS moyenne,
  ROUND(MIN(valeur), 2) AS minimum,
  ROUND(MAX(valeur), 2) AS maximum,
  ROUND(STDDEV(valeur), 2) AS ecart_type
FROM variables
GROUP BY nom
ORDER BY nom
""")
statistiques_ml

## 15. Bilan général

Après l'exécution des cellules, le bilan doit répondre à quatre questions :

1. Les données sont-elles assez complètes et régulières ?
2. Quelles variables météo présentent les amplitudes les plus fortes ?
3. Les différences entre départements et saisons sont-elles cohérentes ?
4. Les variables ML sont-elles disponibles sans utiliser d'information future ?

Cette EDA décrit les données, mais ne permet pas encore de conclure sur les
causes des incendies. Il manque toujours une table d'incendies réellement
La cible actuelle est bien le niveau de danger Météo-France à J1 ou J2.